In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from dataset_loaders.colour_mnist import BG_NAMES, FG_NAMES

plt.rcParams["figure.dpi"] = 200

RESULTS = Path("../results/colourmnist")
QUERY = ["digit", "fg", "bg"]
VALUES = {"digit": [str(d) for d in range(10)], "fg": FG_NAMES, "bg": BG_NAMES}
COLOURS = plt.rcParams["axes.prop_cycle"].by_key()["color"]


def query_label(digit, fg, bg):
    """`(3, 1, -1)` -> `3 · green · any`."""
    return " · ".join("any" if v < 0 else VALUES[f][v] for f, v in zip(QUERY, (digit, fg, bg)))

In [ ]:
calibration = pd.read_csv(RESULTS / "colour_calibration.csv")
histogram = pd.read_csv(RESULTS / "colour_calibration_histogram.csv")

# Per query, only the row scoring all free factors jointly: the strictest one.
width = calibration["factor"].str.count(r"\+")
joint = calibration[width == width.groupby([calibration[f] for f in QUERY]).transform("max")]
joint.fillna({"vae": "-", "checkpoint": "-"}).pivot_table(
    index=["vae", "checkpoint", "source"], columns=QUERY, values="value"
)

In [ ]:
def plot_tv(df, title):
    """Bars: total variation per model and arm, coloured by VAE, dont_care hatched. Line: real training images."""
    rows = df[df["source"] != "real"].sort_values(["vae", "checkpoint", "source"])
    labels = rows["checkpoint"] + " · " + rows["source"]

    fig, ax = plt.subplots(figsize=(9, 8))
    for (vae, group), colour in zip(rows.groupby("vae", sort=False), COLOURS):
        bars = ax.bar(labels[group.index], group["value"], color=colour, label=vae)
        for bar, source in zip(bars, group["source"]):
            if source == "dont_care":
                bar.set_hatch("//")
        ax.bar_label(bars, fmt="{:.3f}", padding=3, fontsize="small")

    real = df[df["source"] == "real"]["value"].mean()
    ax.axhline(real, color="black", linestyle="--", label=f"real ({real:.3f})")

    ax.set_title(f"{title}\ntotal variation (lower is better)")
    ax.tick_params(axis="x", labelrotation=45)
    plt.setp(ax.get_xticklabels(), ha="right")
    ax.legend(fontsize="small")
    fig.tight_layout()
    plt.show()

In [ ]:
overview = joint.groupby(["vae", "checkpoint", "source"], dropna=False)["value"].mean().reset_index()
plot_tv(overview, "mean over all queries")

In [ ]:
for query, df in joint.groupby(QUERY):
    plot_tv(df, f"{query_label(*query)}  (free: {df['factor'].iloc[0]})")

In [ ]:
def plot_histogram(query, checkpoint=None, source="real"):
    """One model arm's mix of each free factor against the training conditional, for one query."""
    df = histogram[
        (histogram[QUERY] == query).all(axis=1)
        & (histogram["source"] == source)
        & ~histogram["factor"].str.contains("+", regex=False)
        ]
    if checkpoint is not None:
        df = df[df["checkpoint"] == checkpoint]
    factors = list(dict.fromkeys(df["factor"]))

    fig, axes = plt.subplots(1, len(factors), figsize=(5 * len(factors), 5), squeeze=False)
    for factor, ax in zip(factors, axes[0]):
        rows = df[df["factor"] == factor]
        x = np.arange(len(rows))
        ax.bar(x - 0.2, rows["truth"], width=0.4, color="tab:gray", label="training")
        ax.bar(x + 0.2, rows["generated"], width=0.4, color="tab:blue", label="generated")
        ax.set_xticks(x, [VALUES[factor][c] for c in rows["class"]], rotation=45, ha="right")
        ax.set_title(factor)
        ax.legend(fontsize="small")
    fig.suptitle(f"{query_label(*query)}: {checkpoint or 'real'} · {source}")
    fig.tight_layout()
    plt.show()

In [ ]:
plot_histogram((-1, -1, -1))
plot_histogram((-1, -1, -1), "cspn_colour_mnist_correlated_anchored_dontcare:v0", "mixture")
plot_histogram((-1, -1, -1), "cspn_colour_mnist_correlated_anchored_dontcare:v0", "dont_care")